# Q-Learning：强化学习的入门课
> 难度标签：高级 | 预计时长：20-30分钟 | 前置知识：无学习经验


> 所属模块：05_强化学习 | 源文件：Q_Learning.py | 核心功能：Off-policy 表格型 TD 学习、Q 表更新、超参数调优

## 概述

Q-Learning 是最经典的强化学习算法，属于无模型（model-free）、离策略（off-policy）的时序差分（TD）方法。它通过一张 Q 表记录每个状态-动作对的价值，用 Bellman 方程迭代更新。

核心更新公式：Q(s,a) <- Q(s,a) + alpha x [r + gamma x max Q(s',a') - Q(s,a)]

"Off-policy" 意味着行为策略（epsilon-greedy，用于探索）和目标策略（greedy，用于评估）是不同的。这让 Q-Learning 可以从任何经验中学习，不受当前探索策略的限制。

脚本在 4x4 网格世界中实现了完整的 Q-Learning 算法，展示了 Q 表、学到的策略和超参数影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from collections import defaultdict

## 数学原理

### 1. 马尔可夫决策过程（MDP）

强化学习的数学框架是 MDP，定义为五元组 $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$：

- $\mathcal{S}$：状态空间
- $\mathcal{A}$：动作空间
- $P(s'|s, a)$：状态转移概率
- $R(s, a)$：奖励函数
- $\gamma \in [0, 1)$：折扣因子

**代码对应**：`gymnasium` 环境封装了 MDP 的所有元素。

### 2. 值函数与 Bellman 方程

**状态值函数**：$V^\pi(s) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty}\gamma^t r_t | s_0 = s\right]$

**动作值函数**：$Q^\pi(s, a) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty}\gamma^t r_t | s_0 = s, a_0 = a\right]$

**Bellman 方程**（最优）：

$$Q^*(s, a) = R(s, a) + \gamma\sum_{s'}P(s'|s, a)\max_{a'}Q^*(s', a')$$

### 3. Q-Learning 算法

**代码对应**：Q-Learning 是**离策略**（off-policy）时序差分学习。

更新规则：

$$Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma\max_{a'}Q(s', a') - Q(s, a)\right]$$

其中 $\alpha$ 为学习率，$r + \gamma\max_{a'}Q(s', a')$ 为 **TD 目标**。

关键：$\max_{a'}Q(s', a')$ 使用**贪婪策略**选择动作，而实际执行的动作由 $\epsilon$-贪婪策略决定（离策略）。

### 4. SARSA 算法

**代码对应**：SARSA 是**同策略**（on-policy）时序差分学习。

更新规则：

$$Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma Q(s', a') - Q(s, a)\right]$$

其中 $a'$ 是**实际执行**的动作（由当前策略选择）。

### 5. Q-Learning vs SARSA

| 特性 | Q-Learning | SARSA |
|------|-----------|-------|
| 策略类型 | 离策略 | 同策略 |
| TD 目标 | $r + \gamma\max_{a'}Q(s', a')$ | $r + \gamma Q(s', a')$ |
| 收敛到 | $Q^*$（最优 Q 函数） | $Q^\pi$（当前策略的 Q 函数） |
| 风险偏好 | 更激进（学习最优） | 更保守（考虑探索风险） |

**代码对应**：Q-Learning 更新中 `np.max(Q[next_state])` vs SARSA 中 `Q[next_state, next_action]`。

### 6. $\epsilon$-贪婪策略

$$\pi(a|s) = \begin{cases} 1 - \epsilon + \epsilon/|\mathcal{A}| & a = \arg\max_{a'}Q(s, a') \\ \epsilon/|\mathcal{A}| & \text{otherwise} \end{cases}$$

$\epsilon$ 控制探索与利用的平衡。通常从 $\epsilon = 1$ 逐步衰减到 $\epsilon_{\min}$。

### 1. 简单网格世界环境

运行 1. 简单网格世界环境 的代码，观察算法在该环节的行为。

In [ ]:
class GridWorld:
    """4x4 网格世界：起点 (0,0)，终点 (3,3)，陷阱 (1,1) 和 (2,2)"""
    def __init__(self, size=4):
        self.size = size
        self.start = (0, 0)
# --- 继续 ---
        self.goal = (3, 3)
        self.traps = {(1, 1), (2, 2)}
        self.actions = [0, 1, 2, 3]  # 上、下、左、右
        self.action_names = ["↑", "↓", "←", "→"]
        self.reset()

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        r, c = self.state
        if action == 0: r = max(0, r - 1)  # 上
        elif action == 1: r = min(self.size - 1, r + 1)  # 下
        elif action == 2: c = max(0, c - 1)  # 左
# --- 条件判断 ---
        elif action == 3: c = min(self.size - 1, c + 1)  # 右

        self.state = (r, c)

        if self.state == self.goal:
            return self.state, 10.0, True  # 到达终点，奖励 +10
        elif self.state in self.traps:
            return self.state, -5.0, True  # 掉入陷阱，奖励 -5
        else:
# --- 返回结果 ---
            return self.state, -0.1, False  # 每步小惩罚，鼓励快速到达

    def render(self, q_table=None):
        symbols = np.full((self.size, self.size), ".", dtype=object)
        symbols[self.goal[0], self.goal[1]] = "G"
        for t in self.traps:
            symbols[t[0], t[1]] = "T"
# --- 条件判断 ---
        if q_table is not None:
            for r in range(self.size):
                for c in range(self.size):
                    if (r, c) not in self.traps and (r, c) != self.goal:
                        q_vals = [q_table.get(((r, c), a), 0) for a in self.actions]
# --- 数值计算 ---
                        symbols[r, c] = self.action_names[np.argmax(q_vals)]
        print("\n".join([" ".join(row) for row in symbols]))

### 2. Q-Learning 算法

运行 2. Q-Learning 算法 的代码，观察算法在该环节的行为。

In [ ]:
def q_learning(env, n_episodes=500, alpha=0.1, gamma=0.99, epsilon=0.1, epsilon_decay=0.995):
    """Q-Learning 核心算法"""
    Q = defaultdict(float)  # Q(s,a) 默认值为 0
    episode_rewards = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False

        while not done:
            # ε-greedy 探索策略
            if np.random.rand() < epsilon:
                action = np.random.choice(env.actions)
            else:
                q_vals = [Q[(state, a)] for a in env.actions]
                action = np.argmax(q_vals)

            next_state, reward, done = env.step(action)
            total_reward += reward

            # Q-Learning 更新（off-policy：用 max Q(s',a') 更新）
            best_next = max(Q[(next_state, a)] for a in env.actions)
            Q[(state, action)] += alpha * (reward + gamma * best_next - Q[(state, action)])

            state = next_state

        episode_rewards.append(total_reward)
        epsilon *= epsilon_decay  # 逐步降低探索率

    return Q, episode_rewards

### 3. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
env = GridWorld()
Q, rewards = q_learning(env, n_episodes=1000, alpha=0.1, gamma=0.99, epsilon=0.2)

print("=== Q-Learning 训练结果 ===")
print(f"最后 100 平均奖励: {np.mean(rewards[-100:]):.2f}")
print(f"收敛时奖励: {np.mean(rewards[-50:]):.2f}")

### 4. 策略展示

运行 4. 策略展示 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 学到的策略 ===")
env.render(Q)

### 5. Q 值表

运行 5. Q 值表 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Q 值表（部分）===")
print(f"{'状态':>10} {'↑':>8} {'↓':>8} {'←':>8} {'→':>8} {'最优':>6}")
for r in range(4):
    for c in range(4):
        state = (r, c)
# --- 条件判断 ---
        if state in env.traps or state == env.goal:
            continue
        q_vals = [Q[(state, a)] for a in env.actions]
        best = env.action_names[np.argmax(q_vals)]
        print(f"  ({r},{c}) {q_vals[0]:>8.3f} {q_vals[1]:>8.3f} {q_vals[2]:>8.3f} "
# --- 继续 ---
              f"{q_vals[3]:>8.3f}   {best}")

### 6. 超参数影响

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 不同超参数对比 ===")
for alpha, gamma, eps in [(0.1, 0.99, 0.2), (0.5, 0.99, 0.2),
                           (0.1, 0.5, 0.2), (0.1, 0.99, 0.01)]:
    Q_test, r_test = q_learning(env, n_episodes=500, alpha=alpha, gamma=gamma, epsilon=eps)
    print(f"  α={alpha}, γ={gamma}, ε={eps}: 最后100平均奖励={np.mean(r_test[-100:]):.2f}")

print("\n=== Q-Learning 要点 ===")
print("- Off-policy：行为策略（ε-greedy）与目标策略（greedy）不同")
print("- Q(s,a) ← Q(s,a) + α × [r + γ×max Q(s',a') - Q(s,a)]")
print("- alpha: 学习率，越大更新越激进")
print("- gamma: 折扣因子，越大越重视远期奖励")
# --- 输出结果 ---
print("- epsilon: 探索率，越大越随机探索")
print("- 优点：简单、可收敛到最优 Q 值")
print("- 缺点：Q 表格大小 = 状态数 × 动作数，不适合连续状态/动作空间")

## 关键代码解释

### Q 表用 defaultdict 存储

```python
Q = defaultdict(float)  # Q(s,a) 默认值为 0
```

用字典而非数组存储 Q 值，适合状态空间较大但大部分未访问的情况。

### Bellman 更新

```python
best_next = max(Q[(next_state, a)] for a in env.actions)
Q[(state, action)] += alpha * (reward + gamma * best_next - Q[(state, action)])
```

关键：max Q(s',a') —— 这就是 off-policy 的体现，取下一状态的最大 Q 值，不管实际选了什么动作。

## 使用示例

```python
Q = defaultdict(float)
state = env.reset()
action = epsilon_greedy(Q, state, epsilon)
next_state, reward, done = env.step(action)
best_next = max(Q[(next_state, a)] for a in env.actions)
Q[(state, action)] += alpha * (reward + gamma * best_next - Q[(state, action)])
```

## 注意事项

1. **Q 表大小 = 状态数 x 动作数**：连续空间不可行，需要 DQN
2. **epsilon 衰减**：初期多探索，后期多利用
3. **alpha 太大**：学习不稳定；太小：收敛太慢
4. **gamma 接近 1**：重视远期回报；接近 0：只看眼前

## 总结与延伸

以上代码展示了 **Q Learning** 的完整流程。

**解读要点**：
- 关注 **累计奖励** 随训练轮次的增长趋势
- 观察 **探索率 epsilon** 的衰减过程
- 对比不同策略（greedy vs epsilon-greedy）的表现

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Double Q-Learning**：用两个 Q 表解决过估计问题
- **DQN**：用神经网络替代 Q 表，处理连续状态空间
- **Q-Learning 的收敛性**：在有限 MDP 下有理论保证，但实际中需要合理的学习率衰减
